# Demo Self-Attention và Causal Mask bằng PyTorch

Bài tập thực hành môn Deep Learning: Hiện thực hóa cơ chế Self-Attention từ đầu bằng PyTorch, trực quan hóa attention heatmap trước và sau Causal Mask, và đo lường chi phí tính toán thực tế trên GPU và CPU.

Dùng **trọng số đã huấn luyện của GPT-2** (bản `gpt2`, 124M tham số), không phải trọng số ngẫu nhiên.

In [1]:
import platform
import torch
from transformers import GPT2Model, GPT2TokenizerFast

torch.manual_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU: Không có (chạy trên CPU)")
print(f"CPU: {platform.processor()}")

C:\Users\Legion\Desktop\TÀI LIỆU HỌC\Presenttation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1113 13:28:34.004000 44460 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


PyTorch version: 2.9.1+cu128
Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
CPU: AMD64 Family 25 Model 97 Stepping 2, AuthenticAMD


## Tải mô hình GPT-2 tiền huấn luyện

Tải mô hình `GPT2Model` và `GPT2TokenizerFast` từ Hugging Face. Mô hình được đặt ở chế độ đánh giá (`eval()`) với kiểu dữ liệu `float32` và `attn_implementation="eager"` để trích xuất ma trận attention gốc làm đối chứng.

In [2]:
print("Đang tải mô hình GPT-2...")
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2Model.from_pretrained("gpt2", attn_implementation="eager")
model.eval()
model.to(DEVICE)
print("Đã tải xong mô hình GPT-2 thành công.")

Đang tải mô hình GPT-2...


C:\Users\Legion\Desktop\TÀI LIỆU HỌC\Presenttation\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Legion\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


C:\Users\Legion\Desktop\TÀI LIỆU HỌC\Presenttation\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Legion\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 74173.38it/s]

Đã tải xong mô hình GPT-2 thành công.
